In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import re

current_year = datetime.now().year
current_month = datetime.now().month
site_name = "루키나호"
base_url = "https://lukina.sunsang24.com/ship/schedule_fleet"

def format_date(raw_date, year):
    try:
        # ✅ 정규식으로 날짜 감지 (연도 포함 또는 미포함)
        match = re.search(r'(\d{1,2})월\s*(\d{1,2})일', raw_date)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            formatted_date = f"{year}-{month:02d}-{day:02d}"
            return formatted_date
        else:
            print(f"❌ 날짜 형식 인식 실패: {raw_date}")
            return None
    except Exception as e:
        print(f"❌ 날짜 형식 변환 실패: {raw_date} - {e}")
        return None
# ✅ 크롤링할 사이트별 메서드 (루키나호)
def crawl_site_lukina(site_name, base_url):
    data = []
    for month in range(current_month, current_month + 12):
        year = current_year if month <= 12 else current_year + 1
        month = month if month <= 12 else month - 12
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        items = soup.select(".shipsinfo_daywarp")
        for item in items:
            raw_date = item.select_one(".date_info").text.strip().replace("\n", "")
            print(f"✅ 원본 날짜: {raw_date}")
            formatted_date = format_date(raw_date, year)  # ✅ 날짜 변환
            
            if not formatted_date:
                print(f"❌ 날짜 변환 실패: {raw_date}")
                continue
            
            wave_power = item.select_one(".date_info2").text.strip()
            zone = "인천권"
            ships = item.select(".small_event_wrap")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>.title").text.strip()
                fish_name = ship.select_one(".fishspecies").text.replace("어종 :", "").split("/")[0].strip() if ship.select_one(".fishspecies") else "[]"
                reservation = ship.select_one(".number.blink_me.n_blue.f_20").text.strip() if ship.select_one(".number.blink_me.n_blue.f_20") else "마감"
                booking_url = f"{base_url}/{year}{month:02d}"
                
                data.append([zone, site_name, ship_name, formatted_date, wave_power, fish_name, reservation, booking_url])
                print(zone, site_name, ship_name, formatted_date, wave_power, fish_name, reservation, booking_url)
    return data
